In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# importing the libraries I need for this question
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
print(f"Shape: {df_delivery.shape}") # print the shape ---
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
df_delivery.info()

In [ ]:
# Task 4: Write your code here:
df_delivery.describe()

In [ ]:
# Task 5: Write your code here:
# Delivery_Time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_delivery = df_delivery.drop(columns=['Order_ID'])

In [ ]:
# checking the data after droping the column
df_delivery.head()

In [ ]:
# Task 2: Write your code here:
# droping the rows where the tagret valuse in null
df_delivery = df_delivery.dropna(subset=['Delivery_Time'])
# Fill categorical columns with 'unknown'
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df_delivery[col] = df_delivery[col].fillna('unknown')
# Fill Courier_Experience_yrs with mode - discrete feature, mode is most representative
df_delivery['Courier_Experience_yrs'] = df_delivery['Courier_Experience_yrs'].fillna(df_delivery['Courier_Experience_yrs'].mode()[0])

# checking the data after dealing with missing values
df_delivery.info()

In [ ]:
df_delivery.head()

In [ ]:
# Task 3: Write your code here:
# checking if ther eare duplicates
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) # change the same dataframe
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

# removing the duplicates
df_delivery.drop_duplicates(inplace=True)

# chceking again after droping
check_duplicates(df_delivery)

In [ ]:
# # Task 4: Write your code here:
# from sklearn.preprocessing import OneHotEncoder

# # Encode categorical columns - converts text to integers
# categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
# for col in categorical_cols:
#     onehot_encoder = OneHotEncoder(sparse_output=False)
#     data_onehot_encoded = onehot_encoder.fit_transform(categorical_cols) # Apply fit_transform to the copied


# data_onehot_encoded.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    # onehot_encoder = OneHotEncoder(sparse_output=False)
    df_delivery[col] = le.fit_transform(df_delivery[col].astype(str))

df_delivery.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df_delivery.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET to prevent data leakage

scaler = StandardScaler()
df_delivery[features] = scaler.fit_transform(df_delivery[features])
df_delivery.head()

In [ ]:
# # Task 6: Write your code here:
# # **Check for target imbalance and state if it is imbalanced or not** (keep this cell empty if not needed)
# def check_target_distribution(df, target_column):
#   df[target_column].hist(bins=30, edgecolor='black')

#   plt.title(f"Target Distribution ({target_column})")
#   plt.xlabel(target_column)
#   plt.ylabel("Frequency")
#   plt.grid(False)

#   plt.show()

# check_target_distribution(df_delivery, "Delivery_Time") #-- chek the skew of the target -- # if yes --> take the log transformor

In [ ]:
# Task 1: Write your code here:
X = df_delivery[features]
y = df_delivery['Delivery_Time']
# print(X)
# print(y)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

n_splits = 5  # K=5 Folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("Random Forest R²:", rf.score(X_test, y_test))

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print("  → Average absolute error")
print("  → Easy to interpret (same units as target)")


# Storage for linear regression results for each fold
lr_losses = []
lr_mae = []

#   # Validate
#   y_pred = np.dot(X_test.values, theta)

#   # Calculate evaluation metrics
#   mae = sklearn_mae(y_test, y_pred)

# # ------baseline in regression is the mean -----#
#   lr_mae.append(mae)

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_
coeffs['Ridge'] = models['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: